# WeVisOne: LLM LoRA + full ViT fine-tune (Colab G4 / A100-80GB)

Same recipe as the local `baseline_s*` runs (LoRA r16/α32 on the LLM, pixel decoder 3e-5, AdamW, 5% warmup + cosine, 230 updates, 5 validations, checkpoint by BUSI val Dice), plus the **whole vision tower** trained with AdamW on fp32 master weights at `--lr_vit 1e-5`.

Before running: upload `wevis_colab_bundle.zip` to `MyDrive/wevis_colab/`. Runtime → Change runtime type → **G4** (or A100). Everything is resumable: re-run all cells after a disconnect; finished runs are skipped. Results land in `MyDrive/wevis_colab/runs/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, subprocess
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{mem:.0f} GB")
assert mem > 38, "Need >= 40 GB (G4 or A100). T4/L4 will run out of memory."


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/wevis_colab')
ROOT = pathlib.Path('/content/wevis')
assert (DRIVE / 'wevis_colab_bundle.zip').exists(), 'upload wevis_colab_bundle.zip to MyDrive/wevis_colab/ first'
if not (ROOT / 'scripts' / 'train.py').exists():
    !mkdir -p /content/wevis && unzip -q /content/drive/MyDrive/wevis_colab/wevis_colab_bundle.zip -d /content/wevis
(DRIVE / 'runs' / 'wevis').mkdir(parents=True, exist_ok=True)
if not (ROOT / 'runs').exists():
    os.symlink(DRIVE / 'runs', ROOT / 'runs')   # checkpoints + eval go straight to Drive
print(sorted(p.name for p in ROOT.iterdir()))


Pinned to the versions used for all local runs (local: Python 3.10; Colab currently Python 3.13, works). Training runs in subprocesses, so no runtime restart is needed.

In [ ]:
!pip -q install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
!pip -q install transformers==5.12.1 peft==0.21.2 accelerate==1.14.0 huggingface-hub==1.21.0 safetensors==0.8.0     qwen_vl_utils==0.0.14 numpy==2.2.6 Pillow==12.2.0 opencv-python-headless==4.10.0.84 scipy pandas einops timm     pycocotools==2.0.11 supervision==0.31.0 matplotlib
# Colab ships torchaudio built for another CUDA (transformers imports it -> peft import fails) and an old torchao
# (peft needs >= 0.16). Same version number, so --force-reinstall is required; torchao is unused here.
!pip -q install --force-reinstall --no-deps torchaudio==2.11.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q uninstall -y torchao
!python -c "import torch, transformers, peft; print(torch.__version__, torch.cuda.is_available(), transformers.__version__, peft.__version__)"


Model: same Hugging Face revision as the local copy; the sha256 check guarantees identical weights.

In [ ]:
MODEL = '/root/models/WeVisOne-VL-4B'
REV = '6b44a45873b4bbc7dca383922504430dc60eca23'
SHA = '002c3ac0783af7c86dc28b282c84dd4bcd0c29dcb3b5e0d584492fe3212acb4a'
if not os.path.exists(f'{MODEL}/model.safetensors'):
    !hf download tencent/WeVisOne-VL-4B --revision {REV} --local-dir {MODEL}
import hashlib
h = hashlib.sha256()
with open(f'{MODEL}/model.safetensors', 'rb') as f:
    for b in iter(lambda: f.read(1 << 24), b''):
        h.update(b)
assert h.hexdigest() == SHA, h.hexdigest()
print('weights verified')


Smoke test (~3 min): 4 updates, checks memory and that the ViT checkpoint saves/loads. Look at `peak_vram_gb` in the VAL line.

In [ ]:
%cd /content/wevis/scripts
import os
os.environ['PYTHONIOENCODING'] = 'utf-8'
S = '../runs/wevis/_smoke_vitfull'
if not os.path.exists(f'{S}/best_vision.pt'):
    rc = os.system(f'python train.py --out {S} --vit_full --max_steps 4 --accum 2 --val_limit 3 > {S}.log 2>&1')
    !grep -E "VAL|params|Error|error" {S}.log | tail -8
    assert rc == 0 and os.path.exists(f'{S}/best_vision.pt'), f'smoke train failed, see {S}.log'
rc = os.system(f'python wevis_eval.py --lora {S}/best_lora --decoder {S}/best_decoder.pt --vision {S}/best_vision.pt --out {S}/eval --sets busi:val --modes generate > {S}_eval.log 2>&1')
assert rc == 0, f'smoke eval failed, see {S}_eval.log'
print('smoke OK')


Main runs: seeds 42/43/44, then evaluation on the same 6 sets as every other model. ~1.5 h per seed on G4.

In [ ]:
ALL = 'busi:val,busi:test,uclm:all,busuc:all,busbra:test,breast:all'
for s in (42, 43, 44):
    run = f'../runs/wevis/vitfull_s{s}'
    if not os.path.exists(f'{run}/TRAIN_DONE'):
        rc = os.system(f'python train.py --out {run} --vit_full --lr_vit 1e-5 --seed {s} --total_steps 230 --n_vals 5 > {run}.log 2>&1')
        assert rc == 0, f'train failed, see {run}.log'
        open(f'{run}/TRAIN_DONE', 'w').close()
    if not os.path.exists(f'{run}/eval/DONE_ALL6'):
        rc = os.system(f'python wevis_eval.py --lora {run}/best_lora --decoder {run}/best_decoder.pt --vision {run}/best_vision.pt '
                       f'--out {run}/eval --sets {ALL} --modes generate > {run}_eval.log 2>&1')
        assert rc == 0, f'eval failed, see {run}_eval.log'
        open(f'{run}/eval/DONE_ALL6', 'w').close()
    print(s, open(f'{run}/best.json').read())


BUSI + BUS-BRA train (1,674 images) with full ViT fine-tune. Matches the local `ms_bra_s*` LoRA runs exactly (strong aug, 690 updates, 4 validations, checkpoint + gate on pooled BUSI+BRA val) so the only difference is the ViT. BUS-BRA test is no longer external here; UCLM, BUS_UC and BrEaST are. ~2–2.5 h per seed on G4.

In [ ]:
MS_SETS = 'busi:val+busbra:val,busi:test,uclm:all,busuc:all,busbra:test,breast:all'
for s in (42, 43, 44):
    run = f'../runs/wevis/ms_bra_vitfull_s{s}'
    if not os.path.exists(f'{run}/TRAIN_DONE'):
        rc = os.system(f'python train.py --out {run} --vit_full --lr_vit 1e-5 --seed {s} --aug strong '
                       f'--train_spec busi:train+busbra:train --val_spec busi:val+busbra:val --total_steps 690 --n_vals 4 > {run}.log 2>&1')
        assert rc == 0, f'train failed, see {run}.log'
        open(f'{run}/TRAIN_DONE', 'w').close()
    if not os.path.exists(f'{run}/eval/DONE_ALL6'):
        rc = os.system(f'python wevis_eval.py --lora {run}/best_lora --decoder {run}/best_decoder.pt --vision {run}/best_vision.pt '
                       f'--out {run}/eval --sets {MS_SETS} --modes generate > {run}_eval.log 2>&1')
        assert rc == 0, f'eval failed, see {run}_eval.log'
        open(f'{run}/eval/DONE_ALL6', 'w').close()
    print(s, open(f'{run}/best.json').read())


Optional hardware control (~15 compute units): the unchanged local recipe (no ViT training) on this GPU, to show that GPU type does not move the numbers. Set `RUN_CONTROL = True`.

In [ ]:
RUN_CONTROL = False
if RUN_CONTROL:
    run = '../runs/wevis/g4ctrl_s42'
    if not os.path.exists(f'{run}/TRAIN_DONE'):
        assert os.system(f'python train.py --out {run} --seed 42 --total_steps 230 --n_vals 5 > {run}.log 2>&1') == 0
        open(f'{run}/TRAIN_DONE', 'w').close()
    if not os.path.exists(f'{run}/eval/DONE_ALL6'):
        assert os.system(f'python wevis_eval.py --lora {run}/best_lora --decoder {run}/best_decoder.pt --out {run}/eval --sets {ALL} --modes generate > {run}_eval.log 2>&1') == 0
        open(f'{run}/eval/DONE_ALL6', 'w').close()
    print(open(f'{run}/best.json').read())


Done. Download `MyDrive/wevis_colab/runs/wevis/vitfull_s42..44` (and `g4ctrl_s42` if run) into `E:\WeVisOne trainuns\wevis\`, then re-run `analyze.py` locally. The `best_vision.pt` files (~0.8 GB each) are only needed to re-evaluate.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader > ../runs/wevis/colab_gpu.txt
!cat ../runs/wevis/colab_gpu.txt; du -sh ../runs/wevis/vitfull_s* ../runs/wevis/ms_bra_vitfull_s* 2>/dev/null
